# PyGee 作物分类纯 Python 版本：带注释教学版

这个版本保留原 notebook 的整体逻辑，同时把每个 cell 改写成更适合学习和复现的写法。

## 整体流程
1. 配置研究区、样本和时间范围  
2. 预处理 Sentinel-2 / Sentinel-1  
3. 构建 7 期时序影像与指数特征  
4. 在 GEE 中提取样本点时序，再拉到本地 pandas  
5. 清洗数据，拟合三次多项式，得到物候系数  
6. 用 J-M 距离筛选特征，训练随机森林  
7. 把 sklearn 随机森林转回 GEE，在全域逐像元重构物候系数并分类  
8. 平滑、导出、面积统计、长势图、SHAP 解释  

## 读这个 notebook 时要特别注意
- 这里的核心不是“单时相分类”，而是“把 4 月到 10 月的时间变化压缩成物候参数再分类”。
- 三次多项式拟合要求每个样本有完整的 7 期数据，所以云掩膜过严会导致样本被删掉。
- 5.7 有三个版本，它们是三种可选的全域分类方案。实际运行时选一个就够了。
- 5.8 平滑和导出时使用的掩膜，要和你在 5.7 里最终采用的版本保持一致。
- `NDVI_a1` 常被解释成早期增长速率，但它本质上仍然是拟合系数，做农学解释时要谨慎。
- 5.10 面积统计是“本地快速统计版”，适合快速汇报。论文级严格面积建议在等面积投影下重新核算。


## Cell 0：配置研究区、样本资产与时间参数

这一格定义了整个项目的基础输入。  
后面所有函数都会依赖这里的变量名，所以不要随意改名。


In [ ]:
# ==================== Cell 0：基础参数配置 ====================
# 这一格的目标：
# 1. 连接 Earth Engine
# 2. 读取研究区与三类作物样本
# 3. 设置时间范围、时间间隔和抽样数量
# 4. 可选地读取耕地掩膜

import ee
import geemap
import pandas as pd
import numpy as np

# 如果内核重启过，必须重新初始化。
# 这里的 project 要换成你自己的 Earth Engine 项目 ID。
ee.Initialize(project="kindle-400911")

# 研究区：宝清县边界
roi = ee.FeatureCollection("projects/ee-erhbuijnfgqthjngsrjyn/assets/bqx")

def set_crop_properties(feature, landcover, crop_name):
    """
    给样本要素补充两个关键属性：
    1. landcover：数值类别标签，供模型训练使用
    2. crop_name：字符串类别名，便于后续统计和可视化
    """
    return feature.set("landcover", landcover, "crop_name", crop_name)

# 三类作物样本
# 注意：
# 这里默认三个资产内部的几何已经是可信样本区域。
# 后续真正用于建模的是从这些区域随机生成的点。
rice = (
    ee.FeatureCollection("projects/ee-erhbuijnfgqthjngsrjyn/assets/shuidao")
    .map(lambda f: set_crop_properties(f, 1, "rice"))
)
corn = (
    ee.FeatureCollection("projects/ee-erhbuijnfgqthjngsrjyn/assets/yumi")
    .map(lambda f: set_crop_properties(f, 2, "corn"))
)
soybean = (
    ee.FeatureCollection("projects/ee-erhbuijnfgqthjngsrjyn/assets/dadou")
    .map(lambda f: set_crop_properties(f, 3, "soybean"))
)

# 时间设置
year = 2024
start_date = "2024-04-01"
end_date = "2024-10-31"

# 每 30 天做一期合成
# 以这组参数计算，后面会得到 7 期影像。
interval = 30

# 每类作物随机抽取多少个点
samples_per_crop = 50

# 可选的耕地掩膜
# 注意：
# 1. 这不是必须项，但如果分类范围中包含大量非耕地，建议使用
# 2. 后面 5.7 / 5.8 如果使用了掩膜，这里的变量名要保持一致
cropland = ee.Image("projects/ee-erhbuijnfgqthjngsrjyn/assets/2020gengdi")

print("✅ 基础参数配置完成")
print("研究区已加载：宝清县")
print(f"时间范围：{start_date} 到 {end_date}")
print(f"时间间隔：每 {interval} 天一期")
print(f"每类随机样本点数量：{samples_per_crop}")


## Cell 1：影像预处理函数

这一格把影像清洗的规则封装成函数。  
后面生成时间序列时会直接调用这些函数。


In [ ]:
# ==================== Cell 1：影像预处理函数 ====================
# 这一格定义四个函数：
# 1. mask_s2_clouds：Sentinel-2 云与阴影掩膜
# 2. preprocess_s1：Sentinel-1 预处理并添加 dB 与比值特征
# 3. get_s2_collection：获取指定时段的 S2 集合
# 4. get_s1_collection：获取指定时段的 S1 集合

def mask_s2_clouds(image):
    """
    Sentinel-2 云处理：
    - 先用 QA60 去掉云和卷云
    - 再结合 MSK_CLDPRB 与 SCL 去掉高云概率、阴影、卷云
    - 最后把光学波段缩放到反射率量级
    """
    qa = image.select("QA60")
    cloud_bit_mask = 1 << 10
    cirrus_bit_mask = 1 << 11

    # 注意：
    # Earth Engine Python API 里逻辑运算常写成 .And() / .Or() / .Not()
    qa_mask = (
        qa.bitwiseAnd(cloud_bit_mask).eq(0)
        .And(qa.bitwiseAnd(cirrus_bit_mask).eq(0))
    )

    cloud_prob = image.select("MSK_CLDPRB")
    scl = image.select("SCL")

    cloud_ok = cloud_prob.lte(30)
    shadow = scl.eq(3)
    cirrus = scl.eq(10)

    scl_mask = cloud_ok.And(cirrus.neq(1)).And(shadow.neq(1))

    # Sentinel-2 SR 常见缩放系数为 0.0001
    optical_bands = image.select("B.*").multiply(0.0001)

    return (
        image.addBands(optical_bands, overwrite=True)
        .updateMask(qa_mask)
        .updateMask(scl_mask)
        .copyProperties(image, ["system:time_start"])
    )

def preprocess_s1(image):
    """
    Sentinel-1 预处理：
    - 去掉无效 looks
    - 增加 VV_db、VH_db
    - 增加 VV/VH 比值
    """
    image = image.updateMask(image.select("numberOfLooks").gt(0))

    db = ee.Image(10.0).multiply(
        image.select(["VV", "VH"]).log10()
    ).rename(["VV_db", "VH_db"])

    ratio = image.select("VV").divide(image.select("VH")).rename("VV_VH_ratio")

    return image.addBands(db).addBands(ratio)

def get_s2_collection(start, end):
    """
    获取指定时段的 Sentinel-2 集合。
    最终只保留后面会用到的 6 个光学波段，并重命名成易读名称。
    """
    return (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(roi)
        .filterDate(start, end)
        .map(mask_s2_clouds)
        .select(
            ["B2", "B3", "B4", "B8", "B11", "B12"],
            ["blue", "green", "red", "nir", "swir1", "swir2"],
        )
    )

def get_s1_collection(start, end):
    """
    获取指定时段的 Sentinel-1 集合。
    这里保留：
    - VV / VH 双极化
    - IW 模式
    - 下降轨
    """
    return (
        ee.ImageCollection("COPERNICUS/S1_GRD")
        .filterBounds(roi)
        .filterDate(start, end)
        .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
        .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VH"))
        .filter(ee.Filter.eq("instrumentMode", "IW"))
        .filter(ee.Filter.eq("orbitProperties_pass", "DESCENDING"))
        .map(preprocess_s1)
        .select(["VV", "VH", "VV_db", "VH_db", "VV_VH_ratio"])
    )

print("✅ 预处理函数已定义")


## Cell 2：构建指数特征与 7 期时间序列

这一格是整个项目的基础层。  
它会把 4 月到 10 月按 30 天切片，每一期做合成，再计算指数特征，并给每一期打上时间偏移标签。


In [ ]:
# ==================== Cell 2：指数计算与时间序列构建 ====================

def cal_vi_sar(image):
    """
    在每一期影像上计算光学指数和 SAR 派生特征。
    这里同时兼容“有 SAR”和“无 SAR”两种情况。
    """
    # ---------- 光学指数 ----------
    mndwi = image.normalizedDifference(["green", "swir1"]).rename("MNDWI")
    ndvi = image.normalizedDifference(["nir", "red"]).rename("NDVI")
    ndwi = image.normalizedDifference(["green", "nir"]).rename("NDWI")

    evi = image.expression(
        "2.5 * (nir - red) / (nir + 6 * red - 7.5 * blue + 1)",
        {
            "red": image.select("red"),
            "nir": image.select("nir"),
            "blue": image.select("blue"),
        },
    ).rename("EVI")

    awei = image.expression(
        "(swir1 - red) / (swir1 + red + 0.16 * green - 0.38 * blue)",
        {
            "red": image.select("red"),
            "swir1": image.select("swir1"),
            "green": image.select("green"),
            "blue": image.select("blue"),
        },
    ).rename("AWEI")

    ndsvi = image.normalizedDifference(["swir1", "red"]).rename("NDSVI")
    ndti = image.normalizedDifference(["swir1", "swir2"]).rename("NDTI")
    gcvi = image.expression(
        "(nir / green) - 1",
        {"nir": image.select("nir"), "green": image.select("green")},
    ).rename("GCVI")
    lswi = image.normalizedDifference(["nir", "swir1"]).rename("LSWI")

    image = image.addBands([mndwi, ndwi, ndvi, evi, awei, ndsvi, ndti, gcvi, lswi])

    # ---------- SAR 派生特征 ----------
    # 注意：
    # 部分时段可能没有可用 S1，所以这里要先判断 VV 是否存在。
    has_sar = image.bandNames().contains("VV")

    sar_bands = ee.Image(
        ee.Algorithms.If(
            has_sar,
            image.select("VV").divide(image.select("VH")).rename("SAR_VV_VH_ratio")
            .addBands(
                image.expression(
                    "(VV - VH) / (VV + VH)",
                    {"VV": image.select("VV"), "VH": image.select("VH")},
                ).rename("SAR_NDVI")
            ),
            # 这里用 -9999 占位，后面在 pandas 阶段转成 NaN
            ee.Image.constant([-9999, -9999]).rename(["SAR_VV_VH_ratio", "SAR_NDVI"]),
        )
    )

    return image.addBands(sar_bands)

def generate_time_series(start_date, end_date, interval):
    """
    把整个生育季切成多个时间窗，每个时间窗生成一张合成影像。
    核心思路：
    - S2 用中值合成
    - S1 若存在则叠加进来
    - 然后计算指数
    - 最后给每一张合成影像附上 period_index 和 time_offset
    """
    start = ee.Date(start_date)
    end = ee.Date(end_date)

    # floor 后得到完整时间窗数量
    n_periods = end.difference(start, "day").divide(interval).floor()

    def process_period(i):
        i = ee.Number(i)

        period_start = start.advance(i.multiply(interval), "day")
        period_end = period_start.advance(interval, "day")

        # 每一期都单独做合成
        s2_composite = get_s2_collection(period_start, period_end).median()
        s1_collection = get_s1_collection(period_start, period_end)
        s1_size = s1_collection.size()

        # 有 S1 就叠加，没有就只保留 S2
        composite = ee.Image(
            ee.Algorithms.If(
                s1_size.gt(0),
                s2_composite.addBands(s1_collection.median()),
                s2_composite,
            )
        )

        composite = cal_vi_sar(composite)

        # 这里把 2024-07-01 作为时间中心
        # 后面三次多项式拟合的自变量 t 就来自这个 time_offset。
        center_date = ee.Date("2024-07-01")
        t = period_start.difference(center_date, "month").round()

        return composite.set(
            {
                "system:time_start": period_start.millis(),
                "period_index": i.add(1),  # 1~7
                "period_start": period_start.format("YYYY-MM-dd"),
                "period_end": period_end.format("YYYY-MM-dd"),
                "time_offset": t,
            }
        )

    time_series_list = ee.List.sequence(0, n_periods.subtract(1)).map(process_period)
    return ee.ImageCollection(time_series_list)

time_series_collection = generate_time_series(start_date, end_date, interval)

print("✅ 时间序列影像集合已生成")
print("提示：后续默认依赖 7 期完整时序。若云太多导致期数不完整，会在清洗阶段丢样本。")


## Cell 3：在 GEE 中抽样并拉回 pandas

这一格把“云端时序影像”转成“本地表格”。  
这是 GEE 与 sklearn 之间的桥接步骤。


In [ ]:
# ==================== Cell 3：样本时序提取与 GEE→Pandas 桥接 ====================

# 需要提取的基础波段与指数
bands_to_extract = [
    "blue", "green", "red", "nir", "swir1", "swir2",
    "NDVI", "EVI", "NDWI", "MNDWI", "LSWI", "GCVI", "NDSVI", "NDTI", "AWEI",
    "VV_db", "VH_db", "SAR_NDVI", "SAR_VV_VH_ratio",
]

def sample_crop_time_series(crop_fc, crop_name, samples_per_period):
    """
    在某一类作物样本区域内随机打点，
    然后对每一期影像提取这些点的特征值。
    """
    # 先把多个样本面 dissolve 成一个整体几何，再随机打点
    crop_geometry = crop_fc.geometry().dissolve(maxError=10)

    sample_points = ee.FeatureCollection.randomPoints(
        region=crop_geometry,
        points=samples_per_period,
        seed=0,  # 固定随机种子，便于复现
    )

    # 给每个点手动补一个 sample_id
    # 注意：
    # 这个 sample_id 在后续按样本点拟合曲线时非常关键。
    sample_list = sample_points.toList(samples_per_period)

    def add_index(i):
        point = ee.Feature(sample_list.get(i))
        return point.set(
            {
                "crop_name": crop_name,
                "landcover": crop_fc.first().get("landcover"),
                "sample_id": i,
            }
        )

    sample_points = ee.FeatureCollection(
        ee.List.sequence(0, samples_per_period - 1).map(add_index)
    )

    def extract_values(img):
        """
        对单期影像进行 reduceRegions。
        输出结果是一批点要素，每个点带有该期的各个波段值。
        """
        period_index = img.get("period_index")
        time_offset = img.get("time_offset")
        period_start = img.get("period_start")

        # 保险做法：
        # 只抽取当前影像里实际存在的波段。
        # 这样即便某期没有 SAR，也不会直接因为 select 报错。
        available_bands = img.bandNames()
        bands_to_select = ee.List(bands_to_extract).filter(
            ee.Filter.inList("item", available_bands)
        )

        reduced = img.select(bands_to_select).reduceRegions(
            collection=sample_points,
            reducer=ee.Reducer.first(),
            scale=10,
            tileScale=4,  # 样本较多时适度增大 tileScale 可减轻内存压力
        )

        def set_period_info(feat):
            return feat.set(
                {
                    "period_index": period_index,
                    "time_offset": time_offset,
                    "period_start": period_start,
                }
            )

        return reduced.map(set_period_info)

    return time_series_collection.map(extract_values).flatten()

print("开始在云端提取样本时序...")
rice_samples = sample_crop_time_series(rice, "rice", samples_per_crop)
corn_samples = sample_crop_time_series(corn, "corn", samples_per_crop)
soybean_samples = sample_crop_time_series(soybean, "soybean", samples_per_crop)

all_samples = rice_samples.merge(corn_samples).merge(soybean_samples)

# ---------- GEE → pandas ----------
print("正在把 GEE FeatureCollection 拉到本地 DataFrame ...")

# 注意：
# 这一行会真正触发计算和下载。
# 样本量增大时，这一步会受网络和本地内存影响。
df = geemap.ee_to_df(all_samples)

# 删除无关系统列
if "system:index" in df.columns:
    df = df.drop(columns=["system:index"])

print("✅ 样本数据已加载到本地")
print(f"DataFrame shape: {df.shape}")
print(df["crop_name"].value_counts())

# 这里可以先看几行，检查 period_index / time_offset / 样本类别是否正常
display(df.head())


## Cell 4：本地分析环境与输出目录

从这一格开始，后半段主要在本地 Python 环境里完成。  
这里集中导入建模、绘图和拟合需要的库。


In [ ]:
# ==================== Cell 4：本地分析依赖与输出路径 ====================

import os
import pandas as pd
import numpy as np
from scipy.optimize import curve_fit
from scipy.interpolate import make_interp_spline
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")

# 中文字体设置
# 注意：
# 这是 Windows 风格配置。如果你在 Linux / macOS 上运行，可能需要换成本地已有字体。
plt.rcParams["font.sans-serif"] = ["SimHei"]
plt.rcParams["axes.unicode_minus"] = False

# 结果输出目录
output_dir = r"C:\Python27\pygee\study\shui"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

print(f"✅ 输出目录已就绪：{output_dir}")


## Cell 5：数据清洗

这一格决定哪些样本能进入拟合和训练。  
这里最重要的两个检查是：

1. SAR 是否足够完整  
2. 每个样本点是否都有完整的 7 期数据


In [ ]:
# ==================== Cell 5：数据清洗 ====================

print("\n" + "=" * 60)
print("开始数据清洗与特征检查...")
print("=" * 60)

# 纯光学特征
spectral_features = [
    "blue", "green", "red", "nir", "swir1", "swir2",
    "NDVI", "EVI", "NDWI", "MNDWI", "LSWI", "GCVI", "NDSVI", "NDTI", "AWEI",
]

# SAR 特征
sar_features = ["VV_db", "VH_db", "SAR_NDVI", "SAR_VV_VH_ratio"]

df_clean = df.copy()

# ---------- 检查 SAR 是否可用 ----------
existing_sar = [col for col in sar_features if col in df_clean.columns]

if existing_sar:
    # 原脚本里用 -9999 表示“这一期没有 SAR”
    # 到 pandas 阶段后，要改成 NaN 才方便做缺失值处理。
    for col in existing_sar:
        df_clean.loc[df_clean[col] == -9999, col] = np.nan

    sar_valid_ratio = df_clean[existing_sar].notna().mean().mean()
    print(f"SAR 整体有效率：{sar_valid_ratio:.2%}")

    if sar_valid_ratio > 0.5:
        print("✅ SAR 覆盖还可以，保留到特征集中")
        spectral_features += existing_sar

        # 注意：
        # 这里采取的是“只保留 SAR 完整的记录”。
        # 好处是保证后续拟合整洁。
        # 代价是样本量可能明显下降。
        df_clean = df_clean.dropna(subset=existing_sar)
    else:
        print("⚠️ SAR 有效率偏低，只用光学特征继续")
        df_clean = df_clean.drop(columns=existing_sar)
else:
    print("⚠️ 数据中没有 SAR 列，只用光学特征继续")

# ---------- 检查每个样本点是否有完整的 7 期 ----------
# 这里的核心逻辑：
# 后面拟合三次多项式时，默认每个样本点都应当有 7 个时相观测。
period_counts = df_clean.groupby(["crop_name", "sample_id"]).size()
incomplete_samples = period_counts[period_counts != 7]

if len(incomplete_samples) > 0:
    print(f"⚠️ 发现 {len(incomplete_samples)} 个样本点时序不完整，将移除")

    complete_sample_ids = period_counts[period_counts == 7].index
    df_clean = df_clean[
        df_clean.set_index(["crop_name", "sample_id"]).index.isin(complete_sample_ids)
    ]

print(f"✅ 清洗完成，剩余记录数：{df_clean.shape[0]}")
print(f"最终纳入拟合的特征列数：{len(spectral_features)}")

# 建议检查一下各类样本是否还比较均衡
display(df_clean.groupby("crop_name")["sample_id"].nunique().rename("完整样本点数量"))


## Cell 6：三次多项式拟合

这是整个项目的核心步骤之一。  
原始 7 期时间序列会被压缩成每个特征的 4 个系数：

- a0：基线水平  
- a1：一次趋势  
- a2：弯曲程度  
- a3：更复杂的时序形态  

同时计算一个 R² 作为拟合质量参考。


In [ ]:
# ==================== Cell 6：三次多项式拟合 ====================

print("\n" + "=" * 60)
print("开始时间序列三次多项式拟合...")
print("=" * 60)

def polynomial_3rd(t, a0, a1, a2, a3):
    """三次多项式：Y = a0 + a1*t + a2*t^2 + a3*t^3"""
    return a0 + a1 * t + a2 * t**2 + a3 * t**3

def fit_time_series_curve(group, features):
    """
    对“单个样本点”的所有特征做拟合。
    group 对应一个 sample_id 的 7 期记录。
    """
    t = group["time_offset"].values

    # 防御性检查
    if len(t) != 7:
        return None

    coefficients = {
        "crop_name": group["crop_name"].iloc[0],
        "landcover": group["landcover"].iloc[0],
        "sample_id": group["sample_id"].iloc[0],
    }

    for feature in features:
        y = group[feature].values

        # 只要某个特征在该样本上存在缺失，就把它的系数都记成 NaN
        if np.isnan(y).any():
            for suffix in ["a0", "a1", "a2", "a3", "r2"]:
                coefficients[f"{feature}_{suffix}"] = np.nan
            continue

        try:
            # maxfev 调大一些，避免非线性拟合迭代次数不足
            popt, _ = curve_fit(polynomial_3rd, t, y, maxfev=10000)

            coefficients[f"{feature}_a0"] = popt[0]
            coefficients[f"{feature}_a1"] = popt[1]
            coefficients[f"{feature}_a2"] = popt[2]
            coefficients[f"{feature}_a3"] = popt[3]

            # 计算拟合优度 R²
            y_pred = polynomial_3rd(t, *popt)
            ss_res = np.sum((y - y_pred) ** 2)
            ss_tot = np.sum((y - np.mean(y)) ** 2)
            coefficients[f"{feature}_r2"] = 1 - (ss_res / ss_tot) if ss_tot != 0 else 0
        except Exception:
            # 拟合失败时，该特征的参数全部设为 NaN
            for suffix in ["a0", "a1", "a2", "a3", "r2"]:
                coefficients[f"{feature}_{suffix}"] = np.nan

    return pd.Series(coefficients)

# groupby 的粒度一定要是“类别 + 标签 + 样本点”
# 这样每一行输出才对应一个样本点的完整物候参数向量。
fitted_params = (
    df_clean.groupby(["crop_name", "landcover", "sample_id"])
    .apply(lambda x: fit_time_series_curve(x, spectral_features))
    .reset_index(drop=True)
)

# 删除拟合后仍有缺失的样本
fitted_params_clean = fitted_params.dropna()

print(f"✅ 拟合完成，有效样本点数：{fitted_params_clean.shape[0]}")
print("各作物拟合成功样本点数量：")
print(fitted_params_clean["crop_name"].value_counts())

# 保存系数表
fitted_params_clean.to_csv(
    os.path.join(output_dir, "fitted_coefficients.csv"),
    index=False,
    encoding="utf-8-sig",
)
print("✅ 拟合系数已保存到 fitted_coefficients.csv")

# 说明：
# 这里得到的不是原始时序，而是“物候参数表”。
# 后面随机森林吃的就是这些参数，而不是 7 期原始值。


## Cell 7：J-M 距离特征筛选

这一格衡量每个物候系数对类别可分性的贡献。  
J-M 距离越大，说明这个特征越能把作物类别分开。


In [ ]:
# ==================== Cell 7：J-M 距离特征筛选 ====================

print("\n" + "=" * 60)
print("计算 J-M 距离进行特征筛选...")
print("=" * 60)

def calculate_jm_distance(class1_data, class2_data):
    """
    计算两类样本在单个特征上的 Jeffries-Matusita 距离。
    经验上数值越接近 2，类别可分性越好。
    """
    mean1, mean2 = class1_data.mean(), class2_data.mean()
    var1, var2 = class1_data.var(), class2_data.var()

    # 方差为 0 时无法正常计算
    if var1 == 0 or var2 == 0 or (var1 + var2) == 0:
        return 0

    B = (
        0.125 * (mean1 - mean2) ** 2 / (0.5 * (var1 + var2))
        + 0.5 * np.log(0.5 * (var1 + var2) / np.sqrt(var1 * var2))
    )

    return min(2 * (1 - np.exp(-B)), 2.0)

# 只拿 a0~a3 进入筛选
# 这里故意不把 r2 当作分类特征。
coefficient_cols = [
    col for col in fitted_params_clean.columns
    if col.endswith(("_a0", "_a1", "_a2", "_a3"))
]

crop_classes = fitted_params_clean["crop_name"].unique()
jm_results = []

for feature in coefficient_cols:
    feature_jm = {"Feature": feature}

    # 两两类别比较
    for i, class1 in enumerate(crop_classes):
        for class2 in crop_classes[i + 1:]:
            data1 = fitted_params_clean.loc[
                fitted_params_clean["crop_name"] == class1, feature
            ]
            data2 = fitted_params_clean.loc[
                fitted_params_clean["crop_name"] == class2, feature
            ]
            feature_jm[f"JM_{class1}_vs_{class2}"] = calculate_jm_distance(data1, data2)

    jm_values = [v for k, v in feature_jm.items() if k.startswith("JM_")]
    feature_jm["JM_mean"] = np.mean(jm_values)
    feature_jm["JM_min"] = np.min(jm_values)
    jm_results.append(feature_jm)

jm_df = pd.DataFrame(jm_results).sort_values("JM_mean", ascending=False)
jm_df.to_csv(
    os.path.join(output_dir, "jm_distance_ranking.csv"),
    index=False,
    encoding="utf-8-sig",
)

# ---------- 动态阈值 ----------
# 这里不把阈值写死到底，而是分层回退。
# 这样特征不会因为阈值过高而少得不够训练。
jm_threshold = 1.8
optimal_features = jm_df.loc[jm_df["JM_mean"] > jm_threshold, "Feature"].tolist()

if len(optimal_features) < 10:
    print(f"⚠️ JM > {jm_threshold} 的特征太少，阈值下调到 1.5")
    jm_threshold = 1.5
    optimal_features = jm_df.loc[jm_df["JM_mean"] > jm_threshold, "Feature"].tolist()

if len(optimal_features) < 5:
    print("⚠️ 特征仍然太少，改为强制使用 Top 30")
    optimal_features = jm_df.head(30)["Feature"].tolist()

print(f"✅ 最终选出 {len(optimal_features)} 个特征")
print("前 10 个示例特征：")
print(optimal_features[:10])

display(jm_df.head(15))


## Cell 8：随机森林训练与评估

这一格开始真正做分类建模。  
输入是上一格筛选出的物候系数，输出是三类作物的分类结果与评估图表。


In [ ]:
# ==================== Cell 8：随机森林训练与评估 ====================

print("\n" + "=" * 60)
print("训练随机森林分类器...")
print("=" * 60)

# 特征与标签
X = fitted_params_clean[optimal_features]
y = fitted_params_clean["landcover"]

# 分层划分训练集 / 测试集，防止类别比例失衡
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y,
)

print(f"训练集样本数：{X_train.shape[0]}")
print(f"测试集样本数：{X_test.shape[0]}")

# 随机森林参数
# 注意：
# 这是一组经验参数，并非唯一最优。
# 真要追求最优结果，可以再做网格搜索或交叉验证。
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=20,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1,
)

rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

oa = accuracy_score(y_test, y_pred)
print(f"\n📊 Overall Accuracy = {oa:.4f} ({oa * 100:.2f}%)")

target_names = [
    f"{name.capitalize()} ({fitted_params_clean.loc[fitted_params_clean['crop_name'] == name, 'landcover'].iloc[0]})"
    for name in crop_classes
]

print("\n📋 分类报告：")
print(classification_report(y_test, y_pred, target_names=target_names))

# ---------- 混淆矩阵 ----------
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(7, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=target_names,
    yticklabels=target_names,
)
plt.title("作物分类混淆矩阵", fontsize=14, fontweight="bold")
plt.ylabel("真实类别")
plt.xlabel("预测类别")
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "confusion_matrix.png"), dpi=300)
plt.show()

# ---------- 特征重要性 ----------
feature_importance = (
    pd.DataFrame(
        {"Feature": optimal_features, "Importance": rf.feature_importances_}
    )
    .sort_values("Importance", ascending=False)
)

feature_importance.to_csv(
    os.path.join(output_dir, "feature_importance.csv"),
    index=False,
)

top_n = min(20, len(optimal_features))

plt.figure(figsize=(10, 6))
plt.barh(
    range(top_n),
    feature_importance.head(top_n)["Importance"].values,
    color="steelblue",
)
plt.yticks(range(top_n), feature_importance.head(top_n)["Feature"].values)
plt.xlabel("Importance")
plt.title(f"Top {top_n} 物候特征重要性", fontweight="bold")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "feature_importance.png"), dpi=300)
plt.show()

print("✅ 模型训练与评估完成")


## Cell 9：三类作物的平均物候曲线

这一格不是做分类，而是帮助你理解原始数据。  
它把每类作物在每一期的均值和标准差画出来，用于观察动态差异。


In [ ]:
# ==================== Cell 9：物候曲线可视化 ====================

print("\n" + "=" * 60)
print("绘制物候特征标准差包络曲线...")
print("=" * 60)

features_to_plot = {
    "NDVI": "NDVI 动态轨迹",
    "NDWI": "NDWI 动态轨迹",
    "GCVI": "GCVI 动态轨迹",
}
crop_colors = {
    "rice": "#FF6B6B",
    "corn": "#4ECDC4",
    "soybean": "#4A90E2",
}

n_features = len(features_to_plot)
fig, axes = plt.subplots(1, n_features, figsize=(6 * n_features, 5))
if n_features == 1:
    axes = [axes]

for idx, (feature, title) in enumerate(features_to_plot.items()):
    ax = axes[idx]

    if feature not in df_clean.columns:
        print(f"⚠️ {feature} 不存在，跳过")
        continue

    for crop in ["rice", "corn", "soybean"]:
        crop_data = df_clean[df_clean["crop_name"] == crop]

        # 先按期统计均值与标准差
        stats = crop_data.groupby("period_index")[feature].agg(["mean", "std"]).reset_index()

        # 把 period_index 映射成大致 DOY
        # 注意：
        # 这里是可视化近似，不是精确到每天的时间戳。
        doy_mapping = {1: 91, 2: 121, 3: 152, 4: 182, 5: 213, 6: 244, 7: 274}
        stats["DOY"] = stats["period_index"].map(doy_mapping)

        x = stats["DOY"].values
        y_mean = stats["mean"].values
        y_std = stats["std"].values

        # 样条平滑仅用于画图美观，不参与任何模型训练
        x_smooth = np.linspace(x.min(), x.max(), 300)
        y_smooth = make_interp_spline(x, y_mean, k=3)(x_smooth)
        y_std_smooth = make_interp_spline(x, y_std, k=3)(x_smooth)

        color = crop_colors[crop]
        ax.plot(
            x_smooth,
            y_smooth,
            color=color,
            linewidth=2.5,
            label=crop.capitalize(),
            zorder=3,
        )
        ax.fill_between(
            x_smooth,
            y_smooth - y_std_smooth,
            y_smooth + y_std_smooth,
            color=color,
            alpha=0.25,
            zorder=2,
        )

    ax.set_xlabel("DOY (Day of Year)", fontsize=12, fontweight="bold")
    ax.set_ylabel("Index value", fontsize=12, fontweight="bold")
    ax.set_title(f"({chr(97 + idx)}) {title}", fontsize=14, fontweight="bold", loc="left")
    ax.set_xlim(90, 300)
    ax.set_xticks([90, 120, 150, 180, 210, 240, 270, 300])
    ax.grid(True, alpha=0.3, linestyle="--")

    if idx == n_features - 1:
        ax.legend(loc="upper right", frameon=True, fontsize=10, edgecolor="black")

plt.tight_layout()
save_path = os.path.join(output_dir, "phenology_curves_combined.png")
plt.savefig(save_path, dpi=300, bbox_inches="tight", facecolor="white")
plt.show()

print(f"✅ 曲线图已保存：{save_path}")
print("说明：这一格用于展示三类作物的平均动态差异，不直接参与分类。")


## Cell 10：个体样本的真实散点与拟合曲线对比

这一格用来检查拟合是否合理。  
它展示某几个样本点的原始观测散点，与三次曲线重构结果。


In [ ]:
# ==================== Cell 10：个体样本拟合对比 ====================

print("\n" + "=" * 60)
print("绘制个体样本的真实散点与拟合曲线对比图...")
print("=" * 60)

import numpy as np
import matplotlib.pyplot as plt
import os

def polynomial_3rd(t, a0, a1, a2, a3):
    return a0 + a1 * t + a2 * t**2 + a3 * t**3

def plot_time_series_comparison(df_raw, df_fitted, feature="NDVI", n_samples=3):
    """
    对每类作物随机抽若干样本点，画出：
    - 原始离散点
    - 三次多项式拟合曲线
    """
    crops = ["rice", "corn", "soybean"]
    fig, axes = plt.subplots(1, len(crops), figsize=(6 * len(crops), 5))

    colors = ["#1f77b4", "#ff7f0e", "#2ca02c"]

    for idx, crop in enumerate(crops):
        ax = axes[idx]
        crop_data = df_fitted[df_fitted["crop_name"] == crop]

        actual_samples = min(n_samples, len(crop_data))
        if actual_samples == 0:
            continue

        # 固定随机种子，便于复现
        sample_ids = crop_data["sample_id"].sample(actual_samples, random_state=42).values

        for i, sample_id in enumerate(sample_ids):
            original = df_raw[
                (df_raw["crop_name"] == crop) & (df_raw["sample_id"] == sample_id)
            ]
            t_raw = original["time_offset"].values
            y_raw = original[feature].values

            params = crop_data[crop_data["sample_id"] == sample_id]
            if params.empty or pd.isna(params[f"{feature}_a0"].values[0]):
                continue

            a0 = params[f"{feature}_a0"].values[0]
            a1 = params[f"{feature}_a1"].values[0]
            a2 = params[f"{feature}_a2"].values[0]
            a3 = params[f"{feature}_a3"].values[0]

            # 拟合曲线是连续曲线，用更密的 t 画出来
            t_fit = np.linspace(-3.2, 3.2, 100)
            y_fit = polynomial_3rd(t_fit, a0, a1, a2, a3)

            color = colors[i % len(colors)]
            ax.plot(
                t_fit,
                y_fit,
                linewidth=2.5,
                alpha=0.85,
                color=color,
                label=f"样本 {sample_id}",
            )
            ax.scatter(
                t_raw,
                y_raw,
                alpha=0.8,
                s=120,
                color=color,
                edgecolors="black",
                linewidth=1.5,
                zorder=5,
            )

        ax.set_title(f"{crop.upper()} - {feature}", fontsize=15, fontweight="bold")
        ax.set_xlabel("距 7 月的时间偏移（月）", fontsize=12)
        ax.set_ylabel(f"{feature} 值", fontsize=12)
        ax.grid(True, alpha=0.3, linestyle="--")

        # x=0 表示 7 月，是拟合中心
        ax.axvline(0, color="#ff4d4d", linestyle="--", alpha=0.8, linewidth=2, label="7 月中心")

        ax.set_xlim(-3.5, 3.5)
        ax.legend(fontsize=10, loc="best")

        for spine in ax.spines.values():
            spine.set_linewidth(1.2)

    plt.tight_layout()

    save_path = os.path.join(output_dir, f"{feature}_time_series_comparison.png")
    plt.savefig(save_path, dpi=300, bbox_inches="tight", facecolor="white")
    print(f"✅ {feature} 对比图已保存：{save_path}")
    plt.show()

# 你关心哪些指数，就放进这个列表
target_features = ["EVI", "NDVI", "LSWI"]

for vi in target_features:
    if vi in df_clean.columns:
        plot_time_series_comparison(df_clean, fitted_params_clean, feature=vi, n_samples=3)
    else:
        print(f"⚠️ 特征 {vi} 不存在，跳过")

print("✅ 样本级拟合效果检查完成")


## Cell 11：全域分类方案 A，不使用耕地掩膜

这是 5.7 的第一个版本。  
适合先验证整条“云端逐像元拟合 + 分类”流程能否跑通。

### 使用建议
- 这是最适合做流程调试的版本
- 若结果里出现大量非耕地误分，可以改跑 Cell 12 或 Cell 13
- 三个 5.7 版本只需要执行一个


In [ ]:
# ==================== Cell 11：5.7 方案 A，不使用耕地掩膜 ====================

print("\n" + "=" * 60)
print("开始进行全域物候特征重构与空间化推演：方案 A（无掩膜）")
print("=" * 60)

import geemap.ml as ml
import ee
import geemap

# ---------- 第一步：把 sklearn 随机森林转成 GEE 分类器 ----------
trees = ml.rf_to_strings(rf, optimal_features)
gee_rf_classifier = ee.Classifier.decisionTreeEnsemble(trees)
print("✅ 本地随机森林已转为 GEE 分类器")

# ---------- 第二步：为每一期影像补上自变量 [1, t, t^2, t^3] ----------
# 这一步的作用，是让后面的逐像元回归能复现本地的三次拟合过程。
base_features_to_fit = list(set([f.split("_a")[0] for f in optimal_features]))
print(f"需要逐像元拟合的基础特征：{base_features_to_fit}")

def add_independent_vars(img):
    t = ee.Image.constant(img.getNumber("time_offset")).rename("t").toFloat()
    t2 = t.pow(2).rename("t2").toFloat()
    t3 = t.pow(3).rename("t3").toFloat()
    constant = ee.Image.constant(1).rename("constant").toFloat()
    return img.addBands([constant, t, t2, t3])

ts_with_x = time_series_collection.map(add_independent_vars)

# ---------- 第三步：逐像元拟合 a0 / a1 / a2 / a3 ----------
# 注意：
# 这是整个云端推理最耗算力的部分。
# 每个基础特征都要单独做一次线性回归求系数。
fitted_bands_list = []

for bf in base_features_to_fit:
    xy_bands = ["constant", "t", "t2", "t3", bf]

    def select_xy(img):
        return img.select(xy_bands)

    xy_collection = ts_with_x.map(select_xy)

    regression = xy_collection.reduce(
        ee.Reducer.linearRegression(numX=4, numY=1)
    )

    # coefficients 输出是 4x1 的数组，需要先降维，再展开成普通波段
    coef_1d = regression.select("coefficients").arrayProject([0])

    coef_band_names = [f"{bf}_a0", f"{bf}_a1", f"{bf}_a2", f"{bf}_a3"]
    coef_image = coef_1d.arrayFlatten([coef_band_names])

    fitted_bands_list.append(coef_image)

# 把每个基础特征的四个系数拼起来
full_fitted_image = ee.ImageCollection(fitted_bands_list).toBands()

# toBands 会自动加前缀 0_ / 1_ / 2_ ...
# 这里统一去掉
new_band_names = full_fitted_image.bandNames().map(
    lambda name: ee.String(name).replace("^\\d+_", "")
)
full_fitted_image = full_fitted_image.rename(new_band_names)

# ---------- 第四步：执行分类 ----------
classification_input = full_fitted_image.select(optimal_features)

# 这里故意不使用耕地掩膜
# 适合先检查流程能不能跑通
classified_image = classification_input.classify(gee_rf_classifier)

# ---------- 第五步：渲染 ----------
Map = geemap.Map()
Map.centerObject(roi, 10)

vis_params = {
    "min": 1,
    "max": 3,
    "palette": ["#FF6B6B", "#4ECDC4", "#4A90E2"],
}

Map.addLayer(roi, {"color": "black"}, "研究区边界", False, 0.5)
Map.addLayer(classified_image.clip(roi), vis_params, "方案 A：分类结果（无掩膜）")

legend_dict = {
    "Rice (水稻)": "#FF6B6B",
    "Corn (玉米)": "#4ECDC4",
    "Soybean (大豆)": "#4A90E2",
}
Map.add_legend(title="作物类型", legend_dict=legend_dict)

print("✅ 方案 A 已完成")
print("注意：若非耕地误分明显，改用 Cell 12 或 Cell 13。")
Map


## Cell 12：全域分类方案 B，使用自有耕地掩膜

这是 5.7 的第二个版本。  
如果你的自建耕地资产质量较好，这通常是更实用的方案。

### 使用建议
- 若你的 `2020gengdi` 范围、投影、取值都可靠，优先用这个
- 后面 5.8 平滑和导出时，掩膜变量也应继续使用 `cropland.gt(0)`
- 三个 5.7 版本只需要执行一个


In [ ]:
# ==================== Cell 12：5.7 方案 B，使用自有耕地掩膜 ====================

print("\n" + "=" * 60)
print("开始进行全域物候特征重构与空间化推演：方案 B（自有耕地掩膜）")
print("=" * 60)

import geemap.ml as ml
import ee
import geemap

trees = ml.rf_to_strings(rf, optimal_features)
gee_rf_classifier = ee.Classifier.decisionTreeEnsemble(trees)
print("✅ 本地随机森林已转为 GEE 分类器")

base_features_to_fit = list(set([f.split("_a")[0] for f in optimal_features]))
print(f"需要逐像元拟合的基础特征：{base_features_to_fit}")

def add_independent_vars(img):
    t = ee.Image.constant(img.getNumber("time_offset")).rename("t").toFloat()
    t2 = t.pow(2).rename("t2").toFloat()
    t3 = t.pow(3).rename("t3").toFloat()
    constant = ee.Image.constant(1).rename("constant").toFloat()
    return img.addBands([constant, t, t2, t3])

ts_with_x = time_series_collection.map(add_independent_vars)

fitted_bands_list = []

for bf in base_features_to_fit:
    xy_bands = ["constant", "t", "t2", "t3", bf]

    def select_xy(img):
        return img.select(xy_bands)

    xy_collection = ts_with_x.map(select_xy)

    regression = xy_collection.reduce(
        ee.Reducer.linearRegression(numX=4, numY=1)
    )

    coef_1d = regression.select("coefficients").arrayProject([0])
    coef_band_names = [f"{bf}_a0", f"{bf}_a1", f"{bf}_a2", f"{bf}_a3"]
    coef_image = coef_1d.arrayFlatten([coef_band_names])

    fitted_bands_list.append(coef_image)

full_fitted_image = ee.ImageCollection(fitted_bands_list).toBands()

new_band_names = full_fitted_image.bandNames().map(
    lambda name: ee.String(name).replace("^\\d+_", "")
)
full_fitted_image = full_fitted_image.rename(new_band_names)

# ---------- 应用自有耕地掩膜 ----------
classification_input = full_fitted_image.select(optimal_features)

# 注意：
# 这里默认你的耕地资产中“耕地像元 > 0”。
# 若你的耕地影像编码方式不同，需要改这里的逻辑。
classification_input = classification_input.updateMask(cropland.gt(0))

classified_image = classification_input.classify(gee_rf_classifier)

Map = geemap.Map()
Map.centerObject(roi, 10)

vis_params = {
    "min": 1,
    "max": 3,
    "palette": ["#FF6B6B", "#4ECDC4", "#4A90E2"],
}

Map.addLayer(roi, {"color": "black"}, "研究区边界", False, 0.5)
Map.addLayer(classified_image.clip(roi), vis_params, "方案 B：分类结果（自有耕地掩膜）")

legend_dict = {
    "Rice (水稻)": "#FF6B6B",
    "Corn (玉米)": "#4ECDC4",
    "Soybean (大豆)": "#4A90E2",
}
Map.add_legend(title="作物类型", legend_dict=legend_dict)

print("✅ 方案 B 已完成")
print("提醒：后续 Cell 14 的 final_map 也要保持使用 cropland.gt(0)")
Map


## Cell 13：全域分类方案 C，使用 ESA WorldCover 耕地掩膜

这是 5.7 的第三个版本。  
当自有耕地资产不稳定时，可以用公开产品做一个统一约束。

### 使用建议
- WorldCover v200 中 40 表示 Cropland
- 若研究区耕地定义与 ESA 产品差异较大，结果可能偏保守
- 三个 5.7 版本只需要执行一个


In [ ]:
# ==================== Cell 13：5.7 方案 C，使用 ESA WorldCover 耕地掩膜 ====================

print("\n" + "=" * 60)
print("开始进行全域物候特征重构与空间化推演：方案 C（ESA 耕地掩膜）")
print("=" * 60)

import geemap.ml as ml
import ee
import geemap

trees = ml.rf_to_strings(rf, optimal_features)
gee_rf_classifier = ee.Classifier.decisionTreeEnsemble(trees)
print("✅ 本地随机森林已转为 GEE 分类器")

base_features_to_fit = list(set([f.split("_a")[0] for f in optimal_features]))
print(f"需要逐像元拟合的基础特征：{base_features_to_fit}")

def add_independent_vars(img):
    t = ee.Image.constant(img.getNumber("time_offset")).rename("t").toFloat()
    t2 = t.pow(2).rename("t2").toFloat()
    t3 = t.pow(3).rename("t3").toFloat()
    constant = ee.Image.constant(1).rename("constant").toFloat()
    return img.addBands([constant, t, t2, t3])

ts_with_x = time_series_collection.map(add_independent_vars)

fitted_bands_list = []

for bf in base_features_to_fit:
    xy_bands = ["constant", "t", "t2", "t3", bf]

    def select_xy(img):
        return img.select(xy_bands)

    xy_collection = ts_with_x.map(select_xy)

    regression = xy_collection.reduce(
        ee.Reducer.linearRegression(numX=4, numY=1)
    )

    coef_1d = regression.select("coefficients").arrayProject([0])
    coef_band_names = [f"{bf}_a0", f"{bf}_a1", f"{bf}_a2", f"{bf}_a3"]
    coef_image = coef_1d.arrayFlatten([coef_band_names])

    fitted_bands_list.append(coef_image)

full_fitted_image = ee.ImageCollection(fitted_bands_list).toBands()

new_band_names = full_fitted_image.bandNames().map(
    lambda name: ee.String(name).replace("^\\d+_", "")
)
full_fitted_image = full_fitted_image.rename(new_band_names)

# ---------- 使用 ESA WorldCover 掩膜 ----------
esa_landcover = ee.ImageCollection("ESA/WorldCover/v200").first()

# WorldCover v200 中，40 表示 Cropland
cropland_mask = esa_landcover.eq(40)

classification_input = full_fitted_image.select(optimal_features).updateMask(cropland_mask)
classified_image = classification_input.classify(gee_rf_classifier)

Map = geemap.Map()
Map.centerObject(roi, 10)

vis_params = {
    "min": 1,
    "max": 3,
    "palette": ["#FF6B6B", "#4ECDC4", "#4A90E2"],
}

Map.addLayer(roi, {"color": "black"}, "研究区边界", False, 0.5)
Map.addLayer(classified_image.clip(roi), vis_params, "方案 C：分类结果（ESA 掩膜）")

legend_dict = {
    "Rice (水稻)": "#FF6B6B",
    "Corn (玉米)": "#4ECDC4",
    "Soybean (大豆)": "#4A90E2",
}
Map.add_legend(title="作物类型", legend_dict=legend_dict)

print("✅ 方案 C 已完成")
print("提醒：若后续采用这个版本，Cell 14 里最好也把掩膜改成 cropland_mask。")
Map


## Cell 14：空间平滑与导出

这一格对分类结果做后处理。  
目的主要是去除椒盐噪声，并把最终结果导出为 GeoTIFF。

### 特别注意
这里的掩膜变量要和你在 Cell 12 或 Cell 13 中实际使用的方案一致。  
原脚本默认写的是 `cropland.gt(0)`，若你最终跑的是 ESA 掩膜版本，请手动改成 `cropland_mask`。


In [ ]:
# ==================== Cell 14：空间平滑与导出 ====================

print("\n" + "=" * 60)
print("开始进行空间平滑处理...")
print("=" * 60)

# ---------- 众数滤波 ----------
# 作用：
# 用邻域多数类别替换孤立像元，减少椒盐噪声。
# 半径太大可能会抹平小地块边界，这里 1.5 像素是一个温和设置。
smoothed_image = classified_image.focal_mode(
    radius=1.5,
    kernelType="circle",
    units="pixels",
)

# ---------- 重新套掩膜 ----------
# 注意：
# 滤波后边缘可能稍微外扩，因此建议重新掩膜。
# 这里默认沿用“自有耕地掩膜”版本。
# 如果你前面最终用的是 ESA 掩膜，请改成：
# final_map = smoothed_image.updateMask(cropland_mask)
final_map = smoothed_image.updateMask(cropland.gt(0))

# ---------- 对比显示 ----------
Map2 = geemap.Map()
Map2.centerObject(roi, 10)

vis_params = {
    "min": 1,
    "max": 3,
    "palette": ["#FF6B6B", "#4ECDC4", "#4A90E2"],
}

Map2.addLayer(roi, {"color": "black"}, "宝清县边界", False, 0.5)

Map2.split_map(
    left_layer=geemap.ee_tile_layer(classified_image, vis_params, "原始分类"),
    right_layer=geemap.ee_tile_layer(final_map, vis_params, "平滑后分类"),
)

legend_dict = {
    "Rice (水稻)": "#FF6B6B",
    "Corn (玉米)": "#4ECDC4",
    "Soybean (大豆)": "#4A90E2",
}
Map2.add_legend(title="作物类型", legend_dict=legend_dict)

print("✅ 平滑前后对比图已生成")
display(Map2)

# ---------- 导出到 Google Drive ----------
print("\n正在提交导出任务到 Google Drive ...")

task = ee.batch.Export.image.toDrive(
    image=final_map.clip(roi),
    description="Baoqing_Crop_Map_2024_Smoothed",
    folder="GEE_Outputs",
    fileNamePrefix="Baoqing_Crop_Map_2024_Smoothed",
    region=roi.geometry(),
    scale=10,
    crs="EPSG:4326",
    maxPixels=1e13,
)
task.start()

print("✅ 导出任务已启动")
print("提醒：如果后续需要更严格的面积统计，建议导出后在等面积投影下复算面积。")


## Cell 15：原 notebook 空白

这一格原始 notebook 中为空，可以保留备用。  
你可以把它用作：

- 额外精度评估
- Kappa、F1、Producer's Accuracy、User's Accuracy 补充计算
- 分类结果裁剪到乡镇级统计


In [ ]:
# ==================== Cell 15：预留空白单元 ====================
# 原 notebook 这一格为空。
# 你可以把这里当作补充分析区。

pass


## Cell 16：原 notebook 空白

这一格原始 notebook 中也为空，可以保留备用。  
例如放：

- 不同年份分类结果对比
- 与其他地类产品的交叉核对
- 误分类样本回看


In [ ]:
# ==================== Cell 16：预留空白单元 ====================
# 原 notebook 这一格为空。
# 这里继续保留，方便你后续插入实验。

pass


## Cell 17：本地快速面积统计

这一格从导出的 TIFF 直接统计每类像元数量，再换算面积。  
它的优点是快，适合做报告和快速检查。

### 特别注意
这里假定导出影像是 10 米分辨率，因此直接把每个像元面积写成 100 平方米。  
这对“快速统计”很方便，但严格论文面积建议再做投影检查。


In [ ]:
# ==================== Cell 17：本地快速面积统计 ====================

print("\n" + "=" * 60)
print("读取本地 TIFF 并进行面积统计...")
print("=" * 60)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import rasterio

tif_path = r"C:\Python27\pygee\study\shui\Baoqing_Crop_Map_2024_Smoothed.tif"

if not os.path.exists(tif_path):
    print(f"❌ 文件不存在：{tif_path}")
else:
    with rasterio.open(tif_path) as src:
        img_data = src.read(1)

        # 这里直接采用“10 m × 10 m = 100 m²”的快速换算
        # 适合快速统计，不适合特别严格的面积论文结果。
        pixel_area_sqm = 100.0
        print(f"✅ 影像读取成功，单像元面积按 {pixel_area_sqm} 平方米计算")

    unique, counts = np.unique(img_data, return_counts=True)
    pixel_counts = dict(zip(unique, counts))

    class_mapping = {
        1: "Rice (水稻)",
        2: "Corn (玉米)",
        3: "Soybean (大豆)",
    }

    results = []

    for crop_id, crop_name in class_mapping.items():
        count = pixel_counts.get(crop_id, 0)
        area_sqm = count * pixel_area_sqm
        area_ha = area_sqm / 10000.0
        area_mu = area_ha * 15.0

        results.append(
            {
                "作物类别": crop_name,
                "像素个数": count,
                "面积 (公顷)": round(area_ha, 2),
                "面积 (亩)": round(area_mu, 2),
            }
        )

    df_area = (
        pd.DataFrame(results)
        .sort_values(by="面积 (亩)", ascending=False)
        .reset_index(drop=True)
    )

    print("\n📊 宝清县 2024 年各作物面积统计：")
    display(df_area)

    # ---------- 饼图 ----------
    plt.figure(figsize=(8, 6))

    color_map = {
        "Rice (水稻)": "#FF6B6B",
        "Corn (玉米)": "#4ECDC4",
        "Soybean (大豆)": "#4A90E2",
    }
    colors = [color_map[name] for name in df_area["作物类别"]]

    wedges, texts, autotexts = plt.pie(
        df_area["面积 (亩)"],
        labels=df_area["作物类别"],
        autopct="%1.1f%%",
        colors=colors,
        startangle=140,
        shadow=True,
        textprops={"fontsize": 12, "fontweight": "bold"},
    )

    for autotext in autotexts:
        autotext.set_color("white")

    plt.title("宝清县 2024 年主要作物种植结构占比", fontsize=16, fontweight="bold")
    plt.tight_layout()

    local_output_dir = os.path.dirname(tif_path)
    pie_chart_path = os.path.join(local_output_dir, "Baoqing_Crop_Area_PieChart_Local.png")
    plt.savefig(pie_chart_path, dpi=300, facecolor="white")
    plt.show()

    print(f"✅ 饼图已保存：{pie_chart_path}")
    print("提醒：若要做严格面积统计，请在等面积投影下重新核算像元面积。")


## Cell 18：玉米单作物长势图

这一格把 `NDVI_a1` 作为长势快慢的代理指标，在玉米区域内做空间表达。

### 特别注意
`a1` 是拟合系数，不是直接观测到的生理量。  
它适合作为“早期增长趋势”的近似指标，但做农学解释时要谨慎表述。


In [ ]:
# ==================== Cell 18：玉米 NDVI_a1 长势图 ====================

print("\n" + "=" * 60)
print("正在提取玉米的 NDVI a1 并渲染长势差异图...")
print("=" * 60)

import geemap
import ee

# 先用分类图把玉米像元挑出来
corn_mask = classified_image.eq(2)

# 从“全域物候参数影像”中提取 NDVI_a1
# 再套上玉米掩膜，这样只显示玉米区域
corn_ndvi_a1 = full_fitted_image.select("NDVI_a1").updateMask(corn_mask)

# 注意：
# 原思路可能是按分位数自动拉伸，但大区域统计分位数容易吃内存。
# 这里改用经验阈值，牺牲一点自适应性，换取更稳定的渲染流程。
min_val = 0.0
max_val = 0.25

print(f"✅ 渲染阈值使用经验范围：{min_val} 到 {max_val}")

Map3 = geemap.Map()
Map3.centerObject(roi, 10)

phenology_vis = {
    "min": min_val,
    "max": max_val,
    "palette": ["#d73027", "#fc8d59", "#fee08b", "#d9ef8b", "#91cf60", "#1a9850"],
}

Map3.addLayer(roi, {"color": "black"}, "宝清县边界", False, 0.5)
Map3.addLayer(corn_ndvi_a1.clip(roi), phenology_vis, "玉米 NDVI a1 长势图")

dict_phenology = {
    "长势极好 / 播种早": "#1a9850",
    "长势良好": "#91cf60",
    "长势中等": "#fee08b",
    "长势较差": "#fc8d59",
    "长势极差 / 缺苗冷害": "#d73027",
}
Map3.add_legend(title="玉米早期长势（NDVI a1）", legend_dict=dict_phenology)

display(Map3)

# 导出
print("\n正在提交玉米长势图导出任务...")
task_phenology = ee.batch.Export.image.toDrive(
    image=corn_ndvi_a1.clip(roi),
    description="Baoqing_Corn_GrowthRate_a1_2024",
    folder="GEE_Outputs",
    fileNamePrefix="Baoqing_Corn_GrowthRate_a1_2024",
    region=roi.geometry(),
    scale=10,
    crs="EPSG:4326",
    maxPixels=1e13,
)
task_phenology.start()
print("✅ 玉米长势图导出任务已启动")


## Cell 19：三类作物综合长势图

这一格还是用 `NDVI_a1`，但展示的是全县三大作物的综合长势空间分布。  
适合做总览图，也方便后续在 GIS 软件中按类别再拆分查看。


In [ ]:
# ==================== Cell 19：三类作物综合 NDVI_a1 长势图 ====================

print("\n" + "=" * 60)
print("正在渲染三大作物综合长势图...")
print("=" * 60)

import geemap
import ee

# 分类图中：
# 1 = 水稻，2 = 玉米，3 = 大豆
all_crops_mask = classified_image.gt(0)
rice_mask = classified_image.eq(1)
corn_mask = classified_image.eq(2)
soybean_mask = classified_image.eq(3)

# 全部作物综合图层
all_crops_a1 = full_fitted_image.select("NDVI_a1").updateMask(all_crops_mask)

# 单作物图层，方便交互查看
rice_a1 = full_fitted_image.select("NDVI_a1").updateMask(rice_mask)
corn_a1 = full_fitted_image.select("NDVI_a1").updateMask(corn_mask)
soybean_a1 = full_fitted_image.select("NDVI_a1").updateMask(soybean_mask)

# 保持统一色标，便于跨作物对比
min_val = 0.0
max_val = 0.25

Map_Combined = geemap.Map()
Map_Combined.centerObject(roi, 10)

phenology_vis = {
    "min": min_val,
    "max": max_val,
    "palette": ["#d73027", "#fc8d59", "#fee08b", "#d9ef8b", "#91cf60", "#1a9850"],
}

Map_Combined.addLayer(roi, {"color": "black"}, "宝清县边界", False, 0.5)

# 默认显示全县综合长势
Map_Combined.addLayer(all_crops_a1.clip(roi), phenology_vis, "全县三大作物综合长势", True)

# 单作物图层默认隐藏，按需打开
Map_Combined.addLayer(rice_a1.clip(roi), phenology_vis, "单独查看：水稻长势", False)
Map_Combined.addLayer(corn_a1.clip(roi), phenology_vis, "单独查看：玉米长势", False)
Map_Combined.addLayer(soybean_a1.clip(roi), phenology_vis, "单独查看：大豆长势", False)

dict_phenology = {
    "长势极好 / 发育快": "#1a9850",
    "长势良好": "#91cf60",
    "长势中等": "#fee08b",
    "长势较差": "#fc8d59",
    "长势极差 / 发育受阻": "#d73027",
}
Map_Combined.add_legend(title="作物早期长势（NDVI a1）", legend_dict=dict_phenology)

display(Map_Combined)

print("\n正在提交综合长势图导出任务...")
task_combined_a1 = ee.batch.Export.image.toDrive(
    image=all_crops_a1.clip(roi),
    description="Baoqing_AllCrops_GrowthRate_a1_2024",
    folder="GEE_Outputs",
    fileNamePrefix="Baoqing_AllCrops_GrowthRate_a1_2024",
    region=roi.geometry(),
    scale=10,
    crs="EPSG:4326",
    maxPixels=1e13,
)
task_combined_a1.start()

print("✅ 综合长势图导出任务已启动")
print("提醒：这个图层适合做总览。若要严格解释单一作物长势，最好结合分类图一起看。")


## Cell 20：SHAP 归因分析

这一格做模型解释。  
目标是回答：

1. 模型最依赖哪些物候特征  
2. 这些特征对不同作物类别的输出影响方向如何  

### 特别注意
不同版本的 SHAP 返回值结构不一样，所以这里保留了兼容处理逻辑。


In [ ]:
# ==================== Cell 20：SHAP 归因分析 ====================

print("\n" + "=" * 60)
print("开始执行 SHAP 归因分析...")
print("=" * 60)

import shap
import matplotlib.pyplot as plt
import matplotlib as mpl
import os
import numpy as np

# ---------- 绘图风格 ----------
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["font.size"] = 12
mpl.rcParams["axes.linewidth"] = 1.5
mpl.rcParams["xtick.major.width"] = 1.5
mpl.rcParams["ytick.major.width"] = 1.5
mpl.rcParams["xtick.direction"] = "out"
mpl.rcParams["ytick.direction"] = "out"

# ---------- 计算 SHAP ----------
explainer = shap.TreeExplainer(rf)
print("正在计算 SHAP 值...")
shap_values_raw = explainer.shap_values(X_test)

# 兼容不同版本 SHAP 的输出结构
if isinstance(shap_values_raw, list):
    # 旧版：直接返回按类别拆分的列表
    shap_values_list = shap_values_raw
elif isinstance(shap_values_raw, np.ndarray) and len(shap_values_raw.shape) == 3:
    # 新版：返回 (样本数, 特征数, 类别数)
    shap_values_list = [
        shap_values_raw[:, :, i]
        for i in range(shap_values_raw.shape[2])
    ]
else:
    raise ValueError("未知的 SHAP 返回结构，请检查 shap 版本")

class_names = ["Rice (水稻)", "Corn (玉米)", "Soybean (大豆)"]

# ---------- 图 1：全局重要性 ----------
print("正在生成 SHAP 全局重要性图...")

plt.figure(figsize=(10, 7))
shap.summary_plot(
    shap_values_list,
    X_test,
    plot_type="bar",
    class_names=class_names,
    show=False,
    color=plt.get_cmap("Set2"),
)

ax = plt.gca()
ax.set_xlabel("Mean |SHAP value|", fontsize=13, fontweight="bold")
ax.tick_params(axis="both", which="major", labelsize=11)
plt.title("Global Feature Importance Driven by SHAP", fontsize=15, fontweight="bold", pad=20)
plt.tight_layout()

shap_global_png = os.path.join(output_dir, "Fig1_SHAP_Global_Importance.png")
shap_global_pdf = os.path.join(output_dir, "Fig1_SHAP_Global_Importance.pdf")
plt.savefig(shap_global_png, dpi=600, bbox_inches="tight", facecolor="white")
plt.savefig(shap_global_pdf, format="pdf", bbox_inches="tight")
plt.show()

# ---------- 图 2：每个类别的 beeswarm ----------
print("\n正在生成各类别 SHAP beeswarm 图...")

for i, crop_name in enumerate(class_names):
    plt.figure(figsize=(10, 6))

    shap.summary_plot(
        shap_values_list[i],
        X_test,
        show=False,
        alpha=0.7,
        cmap="coolwarm",
    )

    ax = plt.gca()
    ax.set_xlabel(f"SHAP value for {crop_name}", fontsize=13, fontweight="bold")
    ax.tick_params(axis="both", which="major", labelsize=11)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    plt.title(f"Feature Impact Analysis for {crop_name}", fontsize=15, fontweight="bold", pad=15)
    plt.tight_layout()

    file_prefix = crop_name.split(" ")[0]
    save_png = os.path.join(output_dir, f"Fig2_SHAP_Beeswarm_{file_prefix}.png")
    save_pdf = os.path.join(output_dir, f"Fig2_SHAP_Beeswarm_{file_prefix}.pdf")

    plt.savefig(save_png, dpi=600, bbox_inches="tight", facecolor="white")
    plt.savefig(save_pdf, format="pdf", bbox_inches="tight")
    plt.show()

print("✅ SHAP 分析完成")
print("说明：这一步解释的是模型如何利用物候参数做分类，不是直接解释遥感机理本身。")
